# GRPO Fine-tuning Phi-2 with QLoRA on OASST1

This notebook fine-tunes Microsoft's Phi-2 model using:
- **GRPO (Group Relative Policy Optimization)** from TRL
- **QLoRA** (4-bit quantization + LoRA) for memory efficiency
- **OpenAssistant OASST1** dataset with quality-based rewards

Optimized for Google Colab Free Tier (T4 GPU, 15GB VRAM)

## 1. Install Dependencies

In [ ]:
# Install required packages
%pip install -q transformers>=4.36.0 trl>=0.7.0 peft>=0.7.0 bitsandbytes>=0.41.0
%pip install -q datasets accelerate huggingface_hub
%pip install -q gradio

## 2. Check GPU and Setup

In [ ]:
import torch
import os

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set environment variables for memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 3. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset, Dataset
from huggingface_hub import login, HfApi
import numpy as np
from typing import List, Dict
import gc

## 4. Hugging Face Login

You'll need a Hugging Face token with write access to push the model.
Get your token from: https://huggingface.co/settings/tokens

In [ ]:
# Login to Hugging Face
from google.colab import userdata

try:
    # Try to get token from Colab secrets
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in using Colab secrets!")
except Exception:
    # Manual login
    print("Please enter your Hugging Face token:")
    login()

## 5. Configuration

In [ ]:
# Model and training configuration
MODEL_ID = "microsoft/phi-2"
OUTPUT_DIR = "./grpo_phi2_oasst1"
HF_REPO_NAME = "phi2-grpo-oasst1"  # Change to your desired repo name

# Training hyperparameters (optimized for T4)
MAX_LENGTH = 512  # Reduced from 2048 for memory
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
NUM_GENERATIONS = 2  # GRPO generates multiple responses per prompt

# Dataset configuration
MAX_SAMPLES = 10000  # Limit samples for Colab time constraints
MIN_QUALITY = 0.5  # Minimum quality threshold

## 6. Load and Preprocess OASST1 Dataset

In [ ]:
def extract_quality_reward(labels: Dict) -> float:
    """Extract reward from OASST1 quality labels."""
    if labels is None:
        return 0.5
    
    names = labels.get("name", [])
    values = labels.get("value", [])
    
    quality_score = 0.5
    helpfulness_score = 0.5
    
    for name, value in zip(names, values):
        if name == "quality":
            quality_score = value
        elif name == "helpfulness":
            helpfulness_score = value
    
    return 0.6 * quality_score + 0.4 * helpfulness_score


def format_prompt(text: str) -> str:
    """Format prompt for Phi-2."""
    return f"Instruct: {text}\nOutput:"


def load_and_preprocess_oasst1(max_samples: int = 10000, min_quality: float = 0.5):
    """Load OASST1 and extract prompt-response pairs with rewards."""
    print("Loading OASST1 dataset...")
    dataset = load_dataset("OpenAssistant/oasst1", split="train")
    print(f"Loaded {len(dataset)} messages")
    
    # Build message lookup
    messages = {}
    for item in dataset:
        messages[item["message_id"]] = {
            "text": item["text"],
            "role": item["role"],
            "parent_id": item["parent_id"],
            "labels": item["labels"],
            "lang": item["lang"],
        }
    
    # Extract pairs
    pairs = []
    for msg_id, msg_data in messages.items():
        if msg_data["role"] != "assistant" or msg_data["lang"] != "en":
            continue
        
        parent_id = msg_data["parent_id"]
        if parent_id is None or parent_id not in messages:
            continue
        
        parent = messages[parent_id]
        if parent["role"] != "prompter" or parent["lang"] != "en":
            continue
        
        reward = extract_quality_reward(msg_data["labels"])
        if reward < min_quality:
            continue
        
        pairs.append({
            "prompt": format_prompt(parent["text"]),
            "completion": msg_data["text"],
            "reward": reward,
        })
        
        if len(pairs) >= max_samples:
            break
    
    print(f"Extracted {len(pairs)} prompt-response pairs")
    return pairs

In [ ]:
# Load and preprocess data
pairs = load_and_preprocess_oasst1(max_samples=MAX_SAMPLES, min_quality=MIN_QUALITY)

# Display statistics
rewards = [p["reward"] for p in pairs]
print(f"\nDataset Statistics:")
print(f"  Samples: {len(pairs)}")
print(f"  Reward mean: {np.mean(rewards):.4f}")
print(f"  Reward std: {np.std(rewards):.4f}")

# Show sample
print(f"\nSample:")
print(f"  Prompt: {pairs[0]['prompt'][:100]}...")
print(f"  Response: {pairs[0]['completion'][:100]}...")
print(f"  Reward: {pairs[0]['reward']:.4f}")

## 7. Load Model with QLoRA

In [ ]:
# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # For generation

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print(f"Model loaded! Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "dense"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Prepare Dataset for GRPO

In [ ]:
def filter_by_length(pairs, tokenizer, max_length):
    """Filter pairs that exceed max length."""
    filtered = []
    for pair in pairs:
        full_text = pair["prompt"] + " " + pair["completion"]
        tokens = tokenizer.encode(full_text, add_special_tokens=True)
        if len(tokens) <= max_length:
            filtered.append(pair)
    return filtered

# Filter by length
pairs_filtered = filter_by_length(pairs, tokenizer, MAX_LENGTH)
print(f"Pairs after length filtering: {len(pairs_filtered)} (from {len(pairs)})")

# Create HuggingFace Dataset - GRPO expects 'prompt' field
train_data = {
    "prompt": [p["prompt"] for p in pairs_filtered],
}

train_dataset = Dataset.from_dict(train_data)
print(f"Training dataset size: {len(train_dataset)}")

## 9. Define Reward Function

In [ ]:
# Create a lookup for ground truth data
prompt_to_reward = {p["prompt"]: p["reward"] for p in pairs_filtered}
prompt_to_completion = {p["prompt"]: p["completion"] for p in pairs_filtered}

def reward_function(completions: List[str], prompts: List[str], **kwargs) -> List[float]:
    """
    Reward function for GRPO training.
    
    Combines:
    1. Quality heuristics (length, coherence)
    2. Similarity to ground truth high-quality responses
    """
    rewards = []
    
    for completion, prompt in zip(completions, prompts):
        reward = 0.0
        
        # Length component: prefer 50-300 words
        word_count = len(completion.split())
        if word_count < 10:
            length_score = 0.1
        elif word_count < 50:
            length_score = word_count / 50
        elif word_count <= 300:
            length_score = 1.0
        else:
            length_score = max(0.5, 1.0 - (word_count - 300) / 500)
        
        # Coherence: penalize repetition
        words = completion.lower().split()
        if len(words) > 0:
            unique_ratio = len(set(words)) / len(words)
            coherence_score = min(1.0, unique_ratio * 1.2)
        else:
            coherence_score = 0.0
        
        # Format: check if response seems complete
        format_score = 1.0 if completion.strip().endswith(('.', '!', '?', '"')) else 0.7
        
        # Combine scores
        reward = 0.4 * length_score + 0.4 * coherence_score + 0.2 * format_score
        
        # Scale to [-1, 1] range for GRPO
        reward = 2 * reward - 1
        
        rewards.append(reward)
    
    return rewards

# Test the reward function
test_completions = [
    "This is a short response.",
    "This is a longer response that provides more detail and explanation about the topic at hand, making it more helpful and informative for the user who asked the question."
]
test_prompts = [pairs_filtered[0]["prompt"]] * 2
test_rewards = reward_function(test_completions, test_prompts)
print(f"Test rewards: {test_rewards}")

## 11. Initialize and Run Training

In [ ]:
# GRPO Configuration
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    max_completion_length=MAX_LENGTH - 128,  # Leave room for prompt
    max_prompt_length=128,
    num_generations=NUM_GENERATIONS,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",  # Disable wandb for Colab
)

print("GRPO Config created!")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Generations per prompt: {NUM_GENERATIONS}")

In [ ]:
# Clear GPU memory before training
gc.collect()
torch.cuda.empty_cache()

print(f"GPU Memory before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Initialize GRPO Trainer
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    reward_funcs=reward_function,
)

print("Trainer initialized!")

In [ ]:
# Start training
print("Starting GRPO training...")
print("This may take 4-6 hours on a T4 GPU.")
print("="*50)

trainer.train()

print("="*50)
print("Training completed!")

## 12. Save and Push Model

In [ ]:
# Save the model locally
print("Saving model locally...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# Push to Hugging Face Hub
api = HfApi()

# Get username
user_info = api.whoami()
username = user_info["name"]
repo_id = f"{username}/{HF_REPO_NAME}"

print(f"Pushing model to {repo_id}...")

# Create repo if it doesn't exist
try:
    api.create_repo(repo_id=repo_id, exist_ok=True)
except Exception as e:
    print(f"Repo creation note: {e}")

# Push the adapter
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"Model pushed to https://huggingface.co/{repo_id}")

## 13. Test the Model

In [ ]:
# Test generation
def generate_response(prompt: str, max_new_tokens: int = 256) -> str:
    formatted = f"Instruct: {prompt}\nOutput:"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the output part
    if "Output:" in response:
        response = response.split("Output:")[-1].strip()
    
    return response

# Test prompts
test_prompts = [
    "What is machine learning?",
    "Write a short poem about coding.",
    "Explain the concept of recursion in programming.",
]

print("Testing the fine-tuned model:")
print("="*50)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    response = generate_response(prompt)
    print(f"Response: {response}")
    print("-"*50)

## 14. Create Gradio App Files for HF Space (Comparison UI)

Run this cell to create the app files. The app will show **side-by-side comparison** of Base Phi-2 vs your GRPO Fine-tuned model!

In [ ]:
# Create app.py for Gradio Space - COMPARISON UI (Base vs Fine-tuned)
app_code = f'''import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import time

# Configuration
BASE_MODEL = "microsoft/phi-2"
ADAPTER_MODEL = "{repo_id}"

print("Loading models for comparison...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load BASE model (no fine-tuning)
print("Loading base Phi-2 model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float32, device_map="cpu",
    trust_remote_code=True, low_cpu_mem_usage=True,
)
base_model.eval()

# Load FINE-TUNED model (with LoRA adapter)
print("Loading fine-tuned model...")
finetuned_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float32, device_map="cpu",
    trust_remote_code=True, low_cpu_mem_usage=True,
)

# Load the LoRA adapter
try:
    finetuned_model = PeftModel.from_pretrained(finetuned_model, ADAPTER_MODEL)
    print("LoRA adapter loaded!")
    adapter_loaded = True
except Exception as e:
    print(f"Could not load adapter: {{e}}")
    adapter_loaded = False
finetuned_model.eval()
print("Both models ready!")


def generate_single(model, prompt, max_tokens, temperature):
    formatted = f"Instruct: {{prompt}}\\nOutput:"
    inputs = tokenizer(formatted, return_tensors="pt")
    start = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=True,
                            temperature=temperature, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - start
    resp = tokenizer.decode(out[0], skip_special_tokens=True)
    if "Output:" in resp:
        resp = resp.split("Output:")[-1].strip()
    return resp, elapsed


def compare_models(prompt, max_tokens=200, temperature=0.7):
    if not prompt.strip():
        return "Please enter a prompt.", "", "", ""
    base_resp, base_time = generate_single(base_model, prompt, max_tokens, temperature)
    ft_resp, ft_time = generate_single(finetuned_model, prompt, max_tokens, temperature)
    base_stats = f"Time: {{base_time:.1f}}s | Words: {{len(base_resp.split())}}"
    ft_stats = f"Time: {{ft_time:.1f}}s | Words: {{len(ft_resp.split())}}"
    return base_resp, ft_resp, base_stats, ft_stats


# Gradio Interface - COMPARISON UI
with gr.Blocks(title="Base vs Fine-tuned Comparison", theme=gr.themes.Base()) as demo:
    gr.Markdown("""
    # Base Phi-2 vs GRPO Fine-tuned Comparison
    
    Compare responses from the **base model** and the **GRPO fine-tuned** model side by side!
    
    **Note:** Running on CPU - each response takes 30-60 seconds.
    """)
    
    prompt_input = gr.Textbox(label="Enter your prompt", placeholder="Ask a question...", lines=3)
    
    with gr.Row():
        max_tokens = gr.Slider(50, 400, 200, step=25, label="Max Tokens")
        temperature = gr.Slider(0.1, 1.0, 0.7, step=0.1, label="Temperature")
    
    compare_btn = gr.Button("Compare Responses", variant="primary", size="lg")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Base Phi-2 (No Fine-tuning)")
            base_output = gr.Textbox(label="", lines=10, show_copy_button=True)
            base_stats = gr.Markdown()
        with gr.Column():
            gr.Markdown("### GRPO Fine-tuned")
            ft_output = gr.Textbox(label="", lines=10, show_copy_button=True)
            ft_stats = gr.Markdown()
    
    gr.Examples([
        ["What is machine learning?"],
        ["Explain recursion with an example."],
        ["Write a poem about AI."],
        ["What are the benefits of renewable energy?"],
    ], inputs=prompt_input, label="Try these examples")
    
    compare_btn.click(compare_models, [prompt_input, max_tokens, temperature],
                     [base_output, ft_output, base_stats, ft_stats])

if __name__ == "__main__":
    demo.launch()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("Created app.py")

# Create requirements.txt
requirements = """transformers>=4.36.0
torch
peft>=0.7.0
accelerate
gradio
huggingface_hub
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("Created requirements.txt")
print("\nFiles ready for Hugging Face Space!")

## 15. Deploy to Hugging Face Space

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_file
# Create requirements.txt
requirements = """transformers>=4.36.0
torch
peft>=0.7.0
accelerate
gradio
huggingface_hub
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

# Create a Space
space_id = f"{username}/{HF_REPO_NAME}-demo"

print(f"Creating/Updating Hugging Face Space: {space_id}")
print("=" * 50)

api = HfApi()

# Step 1: Create repo if it doesn't exist
try:
    create_repo(
        repo_id=space_id,
        repo_type="space",
        space_sdk="gradio",
        exist_ok=True,
    )
    print("✓ Space repository ready!")
except Exception as e:
    print(f"Note: {e}")

# Step 2: Upload app.py
print("Uploading app.py...")
try:
    upload_file(
        path_or_fileobj="app.py",
        path_in_repo="app.py",
        repo_id=space_id,
        repo_type="space",
    )
    print("✓ app.py uploaded!")
except Exception as e:
    print(f"✗ Error uploading app.py: {e}")

# Step 3: Upload requirements.txt
print("Uploading requirements.txt...")
try:
    upload_file(
        path_or_fileobj="requirements.txt",
        path_in_repo="requirements.txt",
        repo_id=space_id,
        repo_type="space",
    )
    print("✓ requirements.txt uploaded!")
except Exception as e:
    print(f"✗ Error uploading requirements.txt: {e}")

print("=" * 50)
print(f"✓ Space deployed!")
print(f"")
print(f"🔗 Visit your Space:")
print(f"   https://huggingface.co/spaces/{space_id}")
print(f"")
print(f"Note: It may take 2-5 minutes for the Space to build and start.")

## Done!

Your model has been:
1. Fine-tuned using GRPO with QLoRA
2. Pushed to Hugging Face Hub
3. Deployed as a Gradio Space

You can now query your model at the Space URL above!